In [1]:
from dotenv import load_dotenv
import os
import requests
import boto3
import json
import datetime
from urllib.parse import quote, urlparse


load_dotenv()  # otomatis baca file .env di direktori sekarang
total_jumlah_file = 0

print('START TRANSACTION; -- Mulai transaction')

# 1. Konfigurasi Kredensial
s3 = boto3.client(
    's3',
    endpoint_url=os.getenv("URL_ENDPOINT"),
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv(
        "AWS_SECRET_KEY_ID")
)

def escape_sql_string(value):
    # Escape single quotes by doubling them
    return value.replace("'", "`")


def escape_ampersand(url):
    # Sebelumnya ada return, dan harus dihapus agar semua karakter bisa dicheck.
    # Ini sebenarnya gak dipakai.
    if "&" in url:
        url = url.replace("&", "%26")
    if "☆" in url:
        url = url.replace("☆", "%E2%98%86")
    if " " in url:
        url = url.replace(" ", "%20")
    if "⁇" in url:
        url = url.replace("⁇", "%E2%81%87")
    if "’" in url:
        url = url.replace("’", "%E2%80%99")
    if "⁚" in url:
        url = url.replace("⁚", "%E2%81%9A")
    if "⁄" in url:
        url = url.replace("⁄", "%E2%81%84")

    return url

def escape_url(url):
    # Mengencode karakter khusus lainnya menggunakan urllib
    return quote(url, safe=":/")


def get_artist_from_title(title, list_title_artist):
    """Fungsi untuk mencocokkan judul dan menemukan artis yang sesuai."""
    for item in list_title_artist:
        # Pisahkan berdasarkan ' --- '
        song_title, artist = item.split(' --- ')
        if title.lower() == song_title.lower():  # Cocokkan tanpa memperhatikan huruf besar/kecil
            return artist
    return "Unknown Artist"  # Return "Unknown Artist" string jika tidak ditemukan


def get_match_title_from_name(title, list_index_title):
    """Fungsi untuk mencocokkan index dan menemukan judul yang sesuai."""
    for item in list_index_title:
        # Pisahkan berdasarkan ' --- '
        song_title, artist = item.split(' --- ')
        if title.lower() == song_title.lower():  # Cocokkan tanpa memperhatikan huruf besar/kecil
            return artist
    return "Unknown Title"  # Return "Unknown Title" string jika tidak ditemukan

bucket_name = 'cybeat-sibeux'
# Akhiri dengan slash (/) untuk menandakan folder
folder_target = 'files/music/anisong/10-lof-quest-of-a-lifetime/'

music_path = []

# Daftar pasangan nama lagu dan artis
list_title_artist = []
# Daftar pasangan nama lagu dan index
list_index_title = []
list_cover = []

category = ""
artist = ""
album_artist = ""
album = ''

# Untuk mencari nama file
file_name_key = ""
# Untuk cek jenis file = direktori
file_dir_value = ""
# Untuk cek jenis file = file
file_type_value = ""
is_dynamic_artist = False
is_dynamic_title = False
disc = 0
folder_api_url = ""

# Collecting each row of values
values = []
playlist_value = []

# Starting timestamp for date_added
base_time = datetime.datetime.now().replace(microsecond=0)

# Base SQL INSERT statement template
insert_statement_template = "INSERT INTO music (id_music, category, link_gdrive, title, artist, cover, favorite, date_added, uploader, is_suspicious, disc_number) VALUES \n"
playlist_insert_statement_template = "INSERT INTO `playlist` (`uid`, `name`, `image`, `type`, `author`, `pin`, `date_pin`, `date`, `editable`, `have_disc`) VALUES \n"
# Parameters for the static fields
# 1 = IndoPride
# 2 = 日本の歌
# 3 = Jowo Mletre
# 4 = Worldwide
favorite = "0"

try:
    # 2. Request Data (List Objects)
    response = s3.list_objects_v2(
        Bucket=bucket_name,
        Prefix=folder_target  # Ini filternya (mirip buka folder)
    )

    # 3. Parsing Hasil
    if 'Contents' in response:
        # print(f"File ditemukan di folder '{folder_target}':\n")
        for obj in response['Contents']:
            key = obj['Key']
            size = obj['Size'] # Ukuran dalam bytes
            last_modified = obj['LastModified']
            
            # Membersihkan nama file dari prefix agar terlihat rapi
            nama_file = key.replace(folder_target, "")
            
            # if nama_file: # Mencegah folder itu sendiri ikut terprint
            # print(f"- {key} (Size: {size} bytes) - {last_modified}")
            nama_file = key.replace(" ", "%20")
            # print(f"cdncloudflare/{nama_file}")
            if (".here.txt" in key):
                txt_url = f"https://cdn.sibeux.my.id/{nama_file}"
                response = requests.get(txt_url)
                # Secara eksplisit set encoding ke 'utf-8' (agar teks non-alphabet bisa terbaca)
                response.encoding = 'utf-8'
                content = response.text
                text_content = json.loads(content)
                # Assign value ke variable album
                category = text_content["category"]
                album = text_content['title']
                album = album.replace("'", "\'")
                artist = text_content["author"]
                album_artist = text_content["author"]
                if text_content['disc'] != '0':
                    disc = int(text_content['disc']) - 1
                    if (text_content['1']) != []:
                        is_dynamic_artist = True
                        for i in range(disc+1):
                            list_title_artist.append(
                                text_content[str(i+1)]
                            )
                elif text_content['1'] != []:
                    is_dynamic_artist = True
                    list_title_artist.append(text_content["1"])
                if text_content['music_name_index']['1'] != []:
                    is_dynamic_title = True
                    for i in range(disc+1):
                        list_index_title.append(
                            text_content['music_name_index'][str(i+1)]
                        )
            # Dapatkan cover image dari file yang ada di dalam folder.
            ext = os.path.splitext(key)[1].lower()
            if ext in ['.jpg', '.jpeg', '.png', '.webp'] and "scan" not in key:
                image_url = f"cdncloudflare/{nama_file}"
                list_cover.append(image_url)
            if ext in ['.mp3', '.flac', '.m4a', '.ogg', '.wav', '.acc']:
                music_url = f"cdncloudflare/{nama_file}"
                # cari artist berdasarkan nama file
                if is_dynamic_artist:
                    artist = get_artist_from_title(
                        file_name_without_ext, list_title_artist[x])
                # Cari judul berdasarkan index
                if is_dynamic_title:
                    title_song = get_match_title_from_name(file_name_without_ext, list_index_title[x])
                    title_song = title_song.replace("\'", "\\'")
                # Tambahkan 7 jam menggunakan timedelta
                date_added = base_time + datetime.timedelta(hours=7)
                date_added_str = date_added.strftime("%Y-%m-%d %H:%M:%S")
                # Escape special characters in the values
                artist = escape_sql_string(artist)
                album = escape_sql_string(album)
                date_added_str = escape_sql_string(date_added_str)
                final_title = ""
                file_name_without_ext = os.path.splitext(
                    ((urlparse(key).path).split("/"))[-1])[0]
                if is_dynamic_title == True:
                    final_title = title_song
                else:
                    final_title = file_name_without_ext
                cover_link = list_cover[0]
                path = urlparse(key).path
                parts = path.split("/")

                target = parts[-2]  # folder sebelum nama file
                x = 1
                if (target != 'flac'):
                    x = target
                query = f"(NULL, '{category}', '{music_url}', '{final_title}', '{artist}', '{cover_link}', '{favorite}', '{date_added_str}', 'Cybeat', 'false', '{x}')"
                # music_path.append(music_url)
                values.append(query)

    else:
        print("Folder kosong atau tidak ditemukan.")

except Exception as e:
    print(f"Terjadi Error: {e}")

# Buat query statement untuk insert album
cover_link = list_cover[0]
cover_link = escape_sql_string(cover_link)
have_disc = 0
if disc > 0:
    have_disc = 1
playlist_value.append(
    f"(NULL, '{album}', '{cover_link}', 'album', '{album_artist}', 'false', NULL, NOW(), 'false', {have_disc})"
)

# Joining all values into the final SQL INSERT statement
playlist_insert_statement = playlist_insert_statement_template + \
    ",\n".join(playlist_value) + ";"
insert_statement = insert_statement_template + \
    ",\n".join(values) + ";"

jumlah_file = len(values)
total_jumlah_file += jumlah_file
print(f'\n-- {jumlah_file} files\n') # print jumlah semua file
print(playlist_insert_statement)
print('SET @new_album_id = LAST_INSERT_ID();')
print(insert_statement)
print('SET @first_new_song_id = LAST_INSERT_ID();')
album_music_insert_statement = 'INSERT INTO album_music (id_playlist, id_music) VALUES\n'
album_music_value = []
for index in range(jumlah_file):
    album_music_value.append(f'(@new_album_id, @first_new_song_id + {index})')
print(album_music_insert_statement + \
    ",\n".join(album_music_value) + ";")

print('COMMIT; -- Kalau semua sukses')
# ROLLBACK -- Kalau ada error di tengah2
print(f'\n-- total file: {total_jumlah_file}\n')

START TRANSACTION; -- Mulai transaction
Terjadi Error: SSL validation failed for https://s3.us-west-004.backblazeb2.com/cybeat-sibeux?list-type=2&prefix=files%2Fmusic%2Fanisong%2F10-lof-quest-of-a-lifetime%2F&encoding-type=url [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)


IndexError: list index out of range